# OpenSpiel Bot Testing

Test notebook for:
- `HumanBot` - CLI interaction
- `AlgorithmBot` - Trajectory generation for LLM training
- `play_game()` - Unified game function

In [ ]:
import sys
sys.path.insert(0, '../AffineFoundation/affinetes/environments/openspiel')

In [ ]:
from env import play_game
from game_config import create_game
from agents import GAME_AGENTS
from algorithm_bot import AlgorithmBot
from human_bot import HumanBot

## 1. Test AlgorithmBot - Trajectory Generation

Generate training data using MCTS algorithm

In [ ]:
# Create game (use a simple game for testing)
task_id = 419920143  # This maps to a specific game
# seed = 42

game, config = create_game(task_id)
print(f"Game: {config['game_name']}")
print(f"Num players: {game.num_players()}")

In [ ]:
# Create AlgorithmBot with MCTS
agent = GAME_AGENTS[config['game_name']]()

bot = AlgorithmBot(
    game=game,
    player_id=0,
    agent=agent,
    algorithm="mcts",
    # seed=seed,
)

print(f"Created AlgorithmBot with algorithm: {bot.get_algorithm()}")

In [ ]:
# Play game and generate trajectory
result = play_game(
    bot=bot,
    task_id=task_id,
    # seed=seed,
    opponent="mcts",
    player_id=0,
)

print(f"Game: {result['game_name']}")
print(f"Score: {result['score']}")
print(f"Returns: {result['returns']}")
print(f"Time: {result['time_taken']:.2f}s")

In [ ]:
result['action_history']

In [ ]:
result['observation']

In [ ]:
result['conversation']

In [ ]:
# View conversation (LLM training format)
conversation = result['conversation']
print(f"Conversation length: {len(conversation)} messages\n")

for i, msg in enumerate(conversation[:6]):  # Show first 6 messages
    print(f"[{msg['role'].upper()}]")
    content = msg['content'][:500] + "..." if len(msg['content']) > 500 else msg['content']
    print(content)
    print("-" * 40)

In [ ]:
# View action history
print(f"Action history ({len(result['action_history'])} actions):")
for action in result['action_history'][:10]:
    marker = "*" if action['is_algorithm'] else " "
    print(f"{marker} Player {action['player_id']}: {action['action_str']}")

## 2. Test with Random Algorithm

In [ ]:
# Test with random algorithm
random_bot = AlgorithmBot(
    game=game,
    player_id=0,
    agent=agent,
    algorithm="random",
    seed=seed,
)

result_random = play_game(
    bot=random_bot,
    task_id=task_id,
    seed=seed,
    opponent="random",
)

print(f"Random vs Random:")
print(f"Score: {result_random['score']}")
print(f"Returns: {result_random['returns']}")

## 3. Generate Multiple Trajectories

In [ ]:
# Generate multiple trajectories for training
trajectories = []

for i in range(5):
    game, config = create_game(task_id)
    agent = GAME_AGENTS[config['game_name']]()
    
    bot = AlgorithmBot(
        game=game,
        player_id=i % 2,  # Alternate player positions
        agent=agent,
        algorithm="mcts",
        seed=seed + i,
    )
    
    result = play_game(
        bot=bot,
        task_id=task_id,
        seed=seed + i,
        opponent="random",
        player_id=i % 2,
    )
    
    trajectories.append({
        'conversation': result['conversation'],
        'score': result['score'],
        'returns': result['returns'],
    })
    
    print(f"Trajectory {i+1}: score={result['score']:.2f}, messages={len(result['conversation'])}")

print(f"\nGenerated {len(trajectories)} trajectories")

## 4. Test HumanBot (Interactive)

Uncomment to play interactively

In [ ]:
# Uncomment to play interactively
task_id = 100_000_001  # This maps to a specific game
# seed = 42

game, config = create_game(task_id)
agent = GAME_AGENTS[config['game_name']]()

human_bot = HumanBot(
    game=game,
    player_id=0,
    agent=agent,
)

result = play_game(
    bot=human_bot,
    task_id=task_id,
    # seed=seed,
    opponent="mcts",
)

print(f"\nGame Over!")
print(f"Score: {result['score']}")
print(f"Returns: {result['returns']}")

In [ ]:
result

## 5. Available Games

In [ ]:
# List all available games
print("Available games:")
for name in sorted(GAME_AGENTS.keys()):
    print(f"  - {name}")

## 6. Parallel Games

In [ ]:
from env import play_games_parallel

# Your list of task IDs
task_ids = [509565, 8541600, 10175660, 12535388, 14491948, 16557006, 17271627, 18564981, 19691575, 24131665, 26052778, 27702793, 29372999, 29864608, 32259761, 36349831, 39840861, 40842934, 43360365, 43464712, 46731929, 49040969, 50759867, 52417280, 52471445, 55151837, 56234001, 58835129, 58934218, 59267940, 59608580, 60465394, 64060094, 64263992, 74922431, 79269545, 79355589, 80227094, 80562896, 80585271, 87033967, 88473041, 88981132, 91207499, 97585033, 98381069, 100315312, 100953643, 101152642, 102941060, 104906261, 107858478, 110904416, 112028189, 114368924, 115104045, 116802568, 119524056, 123020160, 124229584, 126718918, 129273321, 132767182, 135227587, 141063077, 147554936, 147629241, 151209791, 155060944, 155123920, 156356382, 160784606, 161137691, 162076668, 166067353, 166565740, 167692981, 168898084, 169566141, 170249359, 178823316, 178999855, 179672899, 180483606, 185515994, 188226657, 188402256, 189683347, 194779809, 196216546, 198224622, 199495136, 207038188, 209240855, 211705983, 211981052, 215326144, 215339963, 216439347, 216740093, 217106305, 217110220, 217289747, 222850747, 223600872, 223736111, 223823846, 223964860, 230037228, 230743614, 234931038, 235684930, 235695114, 237381715, 237439476, 237641006, 237938689, 238878444, 242233758, 243760363, 244797123, 252361214, 254450949, 256276975, 259834978, 271586160, 271770429, 275626699, 276208875, 277733699, 284773795, 288035395, 288217186, 289494401, 291432255, 295430590, 296517987, 296591423, 296922310, 297365445, 297908184, 299435918, 301884358, 302402940, 304250252, 304815676, 307883563, 311713099, 312796947, 315876832, 316322627, 316358233, 324998394, 325612211, 328353046, 328978173, 332911678, 336635313, 337200796, 338254905, 338374428, 348762336, 352326412, 359597917, 366052160, 366143369, 368034674, 373861544, 380361302, 381785823, 383917349, 384503582, 385557268, 388085181, 390411332, 391651756, 392962569, 393236458, 393538100, 394251526, 396916896, 399258515, 401324324, 402284834, 405742513, 406996377, 412016974, 413490031, 413882302, 415781191, 417353400, 419920143, 420798867, 422184774, 423824133, 427813398, 428852496, 430538182, 430991636, 434661822, 436269847, 438048969, 439051049, 444735198, 447323414, 449251192, 449758400, 451915378, 452397140, 459339183, 459529375, 461682283, 462230697, 463333408, 463903795, 469675952, 475282920, 477144172, 486319252, 488100293, 489301271, 489541586, 495688286, 496346395, 497223444, 498599753, 499585863, 502772628, 503665352, 505273573, 506779646, 512228108, 518538452, 523433386, 523693524, 524698260, 526026833, 526476549, 526522113, 529105188, 536964962, 540556141, 546303768, 547303325, 547527291, 552843967, 554192967, 556178561, 560261590, 567050137, 570180624, 570298630, 578328944, 578656570, 580057417, 586660354, 586862511, 589184851, 597654648, 599341641, 619408401, 633551220]  

stats = play_games_parallel(
    task_ids=task_ids,
    algorithm="mcts",      # Your bot's algorithm
    opponent="random",     # Opponent type
    max_workers=8,         # Number of parallel processes
    mcts_simulations=100,  # Lower = faster (default from agent config)
    show_progress=True,
)

print(f"Average Score: {stats['avg_score']:.3f}")
print(f"Win Rate: {stats['win_rate']:.1%}")
print(f"Total Time: {stats['total_time']:.1f}s")